# Fitness curves

Plots the fitness recorded by Oribots training runs. It reads every run under
`output/`, so it shows whatever you have trained locally.

## Setup

The notebook needs the analysis extras (matplotlib + Jupyter), which are kept
out of the core install. Once per environment:

```
.venv.nosync/bin/python -m pip install -c constraints.txt -r requirements-analysis.txt
.venv.nosync/bin/python -m ipykernel install --user --name oribots
```

then open this notebook with the `oribots` kernel.

## Reading the plots

Each run CSV has one row per generation with the eight canonical columns. The
fitness series are `best_fitness`, `worst_fitness`, `best_parent_fitness`,
`best_offspring_fitness`, and `best_ever_fitness`.

**`best_fitness` means different things per algorithm.** For the **EA** it is
the recorded value, the mean clamped to `[0, 1]`. For **CMA-ES** it is the raw
signed optimizer fitness and can be negative. So do not compare an EA curve and
a CMA-ES curve on the same axis without accounting for that. CMA-ES rows also
leave `best_parent_fitness` / `best_offspring_fitness` blank (they have no
parent/offspring split); those show as gaps.

Paths are resolved from the repository root, not the working directory, so this
runs the same regardless of where Jupyter was started.

In [ ]:
import sys
from pathlib import Path

# This notebook lives in output/analysis/, next to the helper module.
sys.path.insert(0, str(Path.cwd()))
import fitness_curves as fc

print("repo root:", fc.repo_root())
runs = fc.find_run_csvs()
print(f"found {len(runs)} run(s):")
for name, path in runs.items():
    print(f"  {name}: {path.relative_to(fc.repo_root())}")

## Every run, one panel each

All fitness series for each run. Empty series (e.g. CMA-ES parent/offspring)
are skipped rather than drawn flat.

In [ ]:
fc.plot_all_runs();

## Best-ever fitness, all runs on one axis

`best_ever_fitness` is the most comparable single curve within one algorithm.
Across algorithms, mind the scale note above.

In [ ]:
fc.best_ever_comparison();

## Inspect one run

Set `run_name` to one of the names printed above to load its columns directly
for ad-hoc analysis.

In [ ]:
runs = fc.find_run_csvs()
if runs:
    run_name = next(iter(runs))  # change to any name printed above
    data = fc.load_run(runs[run_name])
    print(f"{run_name}: {len(data['num_of_generation'])} generations")
    print("columns:", list(data.keys()))
    print("final best_ever_fitness:", data["best_ever_fitness"][-1])
else:
    print("No runs yet. Train one, e.g.:")
    print("  python run.py -r config/simple.py src/ea/main.py output/simple")